# 04 — Inferenza dei Layer Normativi e Heatmap di Ibridità

Questo notebook implementa il cuore metodologico del progetto: **i livelli gerarchici
non sono predefiniti ma emergono dai dati** della specifica materia analizzata.

## Pipeline in quattro fasi

| Fase | Input | Operazione | Output |
|---|---|---|---|
| **A** | Segmenti di ogni atto | LLM 1: descrizione funzionale contestualizzata per segmento | `segments_descriptions.csv` |
| **B** | Descrizioni funzionali | Embedding + UMAP + HDBSCAN → cluster = layer emergenti | — |
| **B.2** | 10 descrizioni representative per cluster | LLM 2: nome del layer | `layer_mapping.csv` |
| **C** | Articoli + layer noti | LLM 3: distribuzione % articolo × layer | `nodes_heatmap.csv` |
| **D** | Matrice per atto | Entropia media → score di ibridità continuo | `nodes_hybridity.csv` |

## Principio metodologico

Il clustering avviene a livello di **segmento** (articolo o considerando), non di atto.
I cluster che emergono rappresentano i livelli gerarchici specifici per quella materia —
quanti siano lo decide l'algoritmo, non l'analista.

Un atto è **ibrido** se i suoi articoli hanno distribuzioni molto diverse tra loro:
alcuni concentrati su livelli apicali, altri su livelli tecnici di dettaglio.
L'ibridità è misurata come **entropia media degli articoli**.

## Output

| File | Contenuto |
|---|---|
| `segments_descriptions.csv` | Una riga per segmento con la descrizione funzionale (LLM 1) |
| `layer_mapping.csv` | Cluster → nome layer con descrizione e rank gerarchico |
| `nodes_heatmap.csv` | Una riga per (celex, articolo) con % per ogni layer |
| `nodes_hybridity.csv` | Una riga per atto con score ibridità e layer dominante |

---

> **Nota sul costo API**: Fase A chiama l'LLM una volta per segmento, Fase C una volta
> per articolo. Con ~200 atti e ~20 segmenti/atto = ordine di 4.000–6.000 chiamate.
> Il checkpointing granulare permette di riprendere da dove si era interrotti.

## 0. Configurazione

**Modifica solo questa cella.** Il resto del notebook gira in automatico.

In [2]:
import re, json
import pandas as pd
from pathlib import Path

out = Path('../data/output/appalti_it')

# ── Regex per segmentare il testo italiano in articoli ────────────────────────
RE_ART_IT = re.compile(
    r'(?m)^[ \t]*Art(?:icolo)?\.?\s+'
    r'(\d+(?:\s*-?\s*(?:bis|ter|quater|quinquies|sexies|septies|octies|novies|decies))?)'
    r'[ \t]*\.?[ \t]*(?:\(([^)\n]{0,120})\))?[ \t]*$',
    re.IGNORECASE
)

def build_segments_it(full_text):
    """Segmenta full_text italiano in articoli."""
    if not full_text or str(full_text) == 'nan':
        return []
    text = str(full_text)
    segs = []
    splits = list(RE_ART_IT.finditer(text))
    if splits:
        pre = text[:splits[0].start()].strip()
        if len(pre) >= 30:
            segs.append({'tipo': 'preambolo_header', 'identificatore': 'preambolo', 'testo': pre[:3000]})
    for i, m in enumerate(splits):
        art_num = m.group(1).strip()
        rubrica = (m.group(2) or '').strip()
        start   = m.start()
        end     = splits[i+1].start() if i+1 < len(splits) else len(text)
        testo   = text[start:end].strip()
        if len(testo) >= 30:
            segs.append({'tipo': 'articolo', 'identificatore': art_num,
                         'testo': testo[:10000], 'rubrica': rubrica})
    return segs


# ── Carica e prepara atti IT ───────────────────────────────────────────────────
df_it_raw = pd.read_csv(out / 'nodes_texts_it.csv')
df_it_raw = df_it_raw.loc[:, ~df_it_raw.columns.duplicated()]

rows_it = []
for _, row in df_it_raw.iterrows():
    full_text = str(row.get('full_text', '')) if pd.notna(row.get('full_text', '')) else ''
    segs      = build_segments_it(full_text)
    n_art     = sum(1 for s in segs if s['tipo'] == 'articolo')
    status    = 'no_text' if not full_text or full_text == 'nan' \
                else ('no_segments' if n_art == 0 else 'ok')
    rows_it.append({
        'Id':           row.get('id',    row.get('slug', '')),
        'Label':        row.get('label', row.get('id',   '')),
        'title':        row.get('titolo', row.get('title', '')),
        'segments':     json.dumps(segs, ensure_ascii=False),
        'text_status':  status,
        'layer_atteso': row.get('layer_atteso', ''),
        'giurisdizione': 'IT',
        'n_articoli':   n_art,
        'quality_score': row.get('quality_score', 0),
    })

df_it = pd.DataFrame(rows_it)
print(f"IT: {len(df_it)} atti  |  ok={df_it[df_it.text_status=='ok'].shape[0]}")


# ── Carica atti EU (segments già pronti dal fetch EUR-Lex) ────────────────────
eu_path = out / 'nodes_texts_eu_appalti.csv'
if eu_path.exists():
    df_eu_raw = pd.read_csv(eu_path)
    rows_eu = []
    for _, row in df_eu_raw.iterrows():
        segs_raw = row.get('segments', '')
        segs     = json.loads(str(segs_raw)) if segs_raw and str(segs_raw) not in ('nan', '[]', '') else []
        n_art    = sum(1 for s in segs if s['tipo'] == 'articolo')
        full_text = str(row.get('full_text', '') or '')
        status   = 'no_text' if not full_text or full_text == 'nan' \
                   else ('no_segments' if n_art == 0 else 'ok')
        rows_eu.append({
            'Id':           row.get('celex', ''),
            'Label':        row.get('celex', ''),
            'title':        row.get('title', ''),
            'segments':     json.dumps(segs, ensure_ascii=False),
            'text_status':  status,
            'layer_atteso': row.get('layer_atteso', ''),
            'giurisdizione': 'EU',
            'n_articoli':   n_art,
            'quality_score': row.get('quality_score', 0),
        })
    df_eu = pd.DataFrame(rows_eu)
    print(f"EU: {len(df_eu)} atti  |  ok={df_eu[df_eu.text_status=='ok'].shape[0]}")
    df = pd.concat([df_it, df_eu], ignore_index=True)
else:
    df = df_it
    print("nodes_texts_eu_appalti.csv non trovato — solo IT")

print(f"Totale: {len(df)}  |  ok={df[df.text_status=='ok'].shape[0]}")
df.to_csv(out / 'nodes_texts.csv', index=False)
print(f"✓ Salvato nodes_texts.csv — colonne: {list(df.columns)}")

IT: 22 atti  |  ok=17
EU: 6 atti  |  ok=4
Totale: 28  |  ok=21
✓ Salvato nodes_texts.csv — colonne: ['Id', 'Label', 'title', 'segments', 'text_status', 'layer_atteso', 'giurisdizione', 'n_articoli', 'quality_score']


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
#  MATERIA  →  stessa cartella scelta nei notebook 02 e 03
# ─────────────────────────────────────────────────────────────────────────────

MATERIA_NAME = "appalti_it"

TEMA_DESCRIZIONE = (
    "Appalti pubblici e contratti pubblici in Italia. "
    "Disciplina delle procedure di gara, dei requisiti delle stazioni appaltanti, "
    "delle concessioni e degli affidamenti di lavori, servizi e forniture pubblici."
)


# ── Modello OpenAI ─────────────────────────────────────────────────────────────
LLM_MODEL = "gpt-5.4-mini"   


# ── Parametri chiamate API ────────────────────────────────────────────────────
LLM_MAX_TOKENS_A   = 300    # Fase A: descrizione livello di astrazione (2-3 frasi)
LLM_MAX_TOKENS_A2  = 800    # Fase A2: JSON distribuzione % Lamfalussy
LLM_MAX_TOKENS_B2  = 400    # Fase B.2: JSON nome + descrizione layer
LLM_MAX_TOKENS_C   = 500    # Fase C: JSON percentuali
LLM_DELAY_SECONDS  = 0.3    # pausa tra chiamate (rispetta il rate limit)
LLM_MAX_RETRIES    = 3      # tentativi in caso di errore transitorio
LLM_RETRY_DELAY    = 5.0    # secondi tra retry


# ── Parametri checkpoint ──────────────────────────────────────────────────────
CHECKPOINT_EVERY   = 100    # segmenti/articoli tra un salvataggio e il successivo


# ── Parametri clustering ──────────────────────────────────────────────────────
EMBEDDING_MODEL     = "all-mpnet-base-v2"
UMAP_N_COMPONENTS   = 10     # dimensioni ridotte prima di HDBSCAN
UMAP_N_NEIGHBORS    = 15
UMAP_MIN_DIST       = 0.0    # 0.0 ottimizza la separazione dei cluster

HDBSCAN_MIN_CLUSTER = 80
HDBSCAN_MIN_SAMPLES = 20
N_REPR_DESCRIPTIONS = 10     # descrizioni representative per il naming

## 1. Import e Percorsi

In [4]:
import os
import json
import math
import time
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=r'C:\Users\claud\Documents\GitHub\eu-law-network-viz\.env')

from openai import OpenAI
import openai

# ── Percorsi ──────────────────────────────────────────────────────────────────
output_path  = os.path.join('..', 'data', 'output', MATERIA_NAME)
Path(output_path).mkdir(parents=True, exist_ok=True)

INPUT_FILE              = os.path.join(output_path, 'nodes_texts.csv')
EDGES_FILE              = os.path.join(output_path, 'edges_focal.csv')
SEGMENTS_DESC_FILE      = os.path.join(output_path, 'segments_descriptions.csv')
LAYER_MAPPING_FILE      = os.path.join(output_path, 'layer_mapping.csv')
NODES_HEATMAP_FILE      = os.path.join(output_path, 'nodes_heatmap.csv')
NODES_HYBRIDITY_FILE    = os.path.join(output_path, 'nodes_hybridity.csv')
HEATMAP_CKPT_FILE       = os.path.join(output_path, 'heatmap_checkpoint.csv')

SEGMENTS_LAMF_FILE      = os.path.join(output_path, 'segments_lamfalussy.csv')
SEGMENTS_LAMF_CKPT_FILE = os.path.join(output_path, 'segments_lamfalussy_checkpoint.csv')
NODES_LAMFALUSSY_FILE   = os.path.join(output_path, 'nodes_lamfalussy.csv')  # articoli filtrati, per viz

print(f"Materia:       {MATERIA_NAME}")
print(f"Input:         {INPUT_FILE}")
print(f"Modello LLM:   {LLM_MODEL}")
print(f"Embedding:     {EMBEDDING_MODEL}")

Materia:       appalti_it
Input:         ..\data\output\appalti_it\nodes_texts.csv
Modello LLM:   gpt-5.4-mini
Embedding:     all-mpnet-base-v2


## 2. Caricamento Dati

In [5]:
import re, json
import pandas as pd

nodes = pd.read_csv(INPUT_FILE)

# ── Check finale ──────────────────────────────────────────────────────────────
print(f"Nodi totali: {len(nodes)}")
nodes_ok   = nodes[nodes['text_status'] == 'ok'].copy()
nodes_fail = nodes[nodes['text_status'] != 'ok'].copy()
print(f"Atti con testo (text_status=ok):  {len(nodes_ok)}")
print(f"Atti senza testo (esclusi):       {len(nodes_fail)}")
if len(nodes_ok):
    print(f"\nDistribuzione per giurisdizione (solo ok):")
    print(nodes_ok['giurisdizione'].value_counts().to_string())
print(f"\nDistribuzione text_status:")
print(nodes['text_status'].value_counts().to_string())

Nodi totali: 28
Atti con testo (text_status=ok):  21
Atti senza testo (esclusi):       7

Distribuzione per giurisdizione (solo ok):
giurisdizione
IT    17
EU     4

Distribuzione text_status:
text_status
ok             21
no_segments     5
no_text         2


## 3. Divisione in Segmenti

La colonna `segments` di ogni atto contiene una lista JSON di segmenti strutturati
(articoli, considerando, allegati). Questa cella costruisce un DataFrame flat
`segments_df` con **una riga per segmento** — l'unità di analisi del clustering.

In [6]:
def parse_segments(row):
    """Parsa la colonna 'segments' e restituisce lista di dict arricchiti."""
    celex = row.get('Label', row['Id'])
    title = str(row.get('title', ''))
    raw   = row.get('segments')

    if pd.isna(raw) or not str(raw).strip():
        return []
    try:
        segs = json.loads(str(raw))
    except (json.JSONDecodeError, ValueError):
        return []

    result = []
    for s in segs:
        result.append({
            'celex':          celex,
            'node_id':        row['Id'],
            'title_atto':     title,
            'tipo':           s.get('tipo', ''),
            'identificatore': s.get('identificatore', ''),
            'testo':          s.get('testo', ''),
        })
    return result


all_segments = []
for _, row in nodes_ok.iterrows():
    all_segments.extend(parse_segments(row))

segments_df = pd.DataFrame(all_segments)

# ID univoco per segmento
segments_df['segment_id'] = (
    segments_df['celex'] + '__' +
    segments_df['tipo'] + '__' +
    segments_df['identificatore'].astype(str)
)

# Rimuove duplicati su segment_id (stesso atto, stesso tipo, stesso identificatore)
before_dedup = len(segments_df)
segments_df = segments_df.drop_duplicates(subset='segment_id', keep='first').reset_index(drop=True)

# Rimuove segmenti con testo troppo breve per essere informativi
MIN_TESTO_LEN = 30
before = len(segments_df)
segments_df = segments_df[segments_df['testo'].str.len() >= MIN_TESTO_LEN].reset_index(drop=True)

# Esclude considerando e header preambolo — non entrano nel clustering né nella heatmap
TIPI_ESCLUSI = {'considerando', 'preambolo_header'}
before_tipi = len(segments_df)
segments_df = segments_df[~segments_df['tipo'].isin(TIPI_ESCLUSI)].reset_index(drop=True)

print(f"Segmenti estratti:              {before_dedup:,}")
print(f"Segmenti scartati (duplicati):  {before_dedup - before:,}")
print(f"Segmenti validi (>={MIN_TESTO_LEN} car): {len(segments_df):,}")
print(f"Segmenti scartati (testo):      {before - before_tipi:,}")
print(f"Segmenti scartati (tipo):       {before_tipi - len(segments_df):,}")
print()
print("Distribuzione per tipo:")
print(segments_df['tipo'].value_counts().to_string())
print()
print(f"Segmenti medi per atto:  {segments_df.groupby('celex').size().mean():.1f}")

Segmenti estratti:              1,626
Segmenti scartati (duplicati):  213
Segmenti validi (>=30 car): 906
Segmenti scartati (testo):      0
Segmenti scartati (tipo):       507

Distribuzione per tipo:
tipo
articolo    906

Segmenti medi per atto:  43.1


## 4. Fase A — Livello di Astrazione per Segmento (LLM 1)

Per ogni segmento l'LLM produce una **descrizione libera del livello di astrazione**:
dove si colloca il segmento nella piramide normativa — quanto è fondazionale vs tecnico/operativo.

La descrizione è volutamente domain-agnostic e non usa la terminologia Lamfalussy:
non vengono imposti tag o categorie. I livelli specifici per materia emergeranno
liberamente dal clustering in Fase B.

**Fase A2** (successiva) assegnerà poi i livelli Lamfalussy standard (L1–L4)
usando queste descrizioni come contesto.

> **Checkpoint**: i risultati vengono salvati in `segments_descriptions.csv` ogni
> `CHECKPOINT_EVERY` segmenti. Rieseguire la cella riprende dal punto di interruzione.

In [7]:
def build_prompt_functional_description(testo, tipo, identificatore, title_atto, tema):
    return f"""You are an expert in Italian and European law, 
    with knowledge of EU legislation and its transposition into Italian law.

Your task is to describe the position of this legal segment in the regulatory
hierarchy — how general or specific it is, and why.

Focus exclusively on the ABSTRACTION LEVEL:
Does this segment state a broad principle that governs the entire framework,
or does it implement a narrow technical detail that only applies in a specific
situation? Where on the spectrum from foundational to operational does it sit?

Describe:
1. Where it sits on the spectrum (foundational / structural / operational / technical)
2. Why — what feature of the text places it there (e.g., it establishes a purpose,
   it delegates a power, it specifies a procedure, it fixes a threshold,
   it sets an effective date, it defines a concept)

## Critical rules
- Do NOT describe what the segment is about thematically.
- Do NOT mention the specific subject matter (FDI, data protection, banking,
  subsidies, etc.).
- Describe only the position in the normative hierarchy and its structural reason.
- 2-3 sentences maximum.

## Examples of correct descriptions
- "Foundational recital that states the overarching policy rationale justifying
   the entire legislative intervention — sits at the highest level of abstraction
   as it frames the purpose of the whole framework without specifying any operative rule."
- "Structural empowerment clause delegating secondary rule-making power to an
   institution within defined substantive limits — sits at an intermediate level,
   organising institutional competences without specifying how they must be exercised."
- "Narrow operative criterion specifying one concrete factor to be checked during
   a particular assessment — highly specific, functioning as a technical sub-rule
   within a broader procedure."
- "Terminal commencement clause fixing the date the act becomes legally effective —
   sits at the most technical end, containing no substantive normative content."

## Also avoid
- Descriptions so vague they say nothing: "Sets out a provision within the framework."
- Restating what the text says without identifying the hierarchical position.
- Any reference to the subject matter of the regulation.

Act title: {title_atto}
Segment ({tipo} {identificatore}):
{testo}

Reply ONLY with the description (2-3 sentences), in English, no other text."""

def call_llm(client, prompt, max_tokens):
    """
    Chiama l'LLM OpenAI con retry automatico su errori transitori.
    Restituisce (testo_risposta, status) dove status è 'ok' | 'error' | 'empty'.
    """
    for attempt in range(LLM_MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                max_completion_tokens=max_tokens,
                temperature=0.3,
                messages=[{'role': 'user', 'content': prompt}]
            )
            text = response.choices[0].message.content.strip()
            if not text:
                return '', 'empty'
            return text, 'ok'

        except openai.RateLimitError:
            wait = LLM_RETRY_DELAY * (attempt + 1) * 2
            print(f"  [RateLimit] attesa {wait:.0f}s (tentativo {attempt+1}/{LLM_MAX_RETRIES})")
            time.sleep(wait)

        except openai.APIStatusError as e:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

        except Exception as e:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

    return 'ERROR: max retries exceeded', 'error'


print("Funzioni LLM definite.")

Funzioni LLM definite.


In [8]:
# ── TEST su 5 segmenti ────────────────────────────────────────────────────────
''' from openai import OpenAI

client = OpenAI()


test_segments = segments_df.sample(5, random_state=42)

for _, seg in test_segments.iterrows():
    prompt = build_prompt_functional_description(
        testo          = seg['testo'],
        tipo           = seg['tipo'],
        identificatore = seg['identificatore'],
        title_atto     = seg['title_atto'],
        tema           = TEMA_DESCRIZIONE,
    )
    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)
    
    print(f"CELEX: {seg['celex']}  |  {seg['tipo']} {seg['identificatore']}")
    print(f"Testo: {seg['testo'][:150]}...")
    print(f"→ {descrizione}")
    print()'''

' from openai import OpenAI\n\nclient = OpenAI()\n\n\ntest_segments = segments_df.sample(5, random_state=42)\n\nfor _, seg in test_segments.iterrows():\n    prompt = build_prompt_functional_description(\n        testo          = seg[\'testo\'],\n        tipo           = seg[\'tipo\'],\n        identificatore = seg[\'identificatore\'],\n        title_atto     = seg[\'title_atto\'],\n        tema           = TEMA_DESCRIZIONE,\n    )\n    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)\n\n    print(f"CELEX: {seg[\'celex\']}  |  {seg[\'tipo\']} {seg[\'identificatore\']}")\n    print(f"Testo: {seg[\'testo\'][:150]}...")\n    print(f"→ {descrizione}")\n    print()'

In [9]:
# ── Gestione checkpoint ───────────────────────────────────────────────────────
if os.path.exists(SEGMENTS_DESC_FILE):
    segs_done = pd.read_csv(SEGMENTS_DESC_FILE)
    done_ids  = set(segs_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_ids):,} segmenti già descritti.")
else:
    segs_done = pd.DataFrame()
    done_ids  = set()
    print("Nessun checkpoint — si parte da zero.")

segments_todo = segments_df[~segments_df['segment_id'].isin(done_ids)].copy()
print(f"Da descrivere: {len(segments_todo):,}")

if len(segments_todo) == 0:
    print("✓ Tutti i segmenti già descritti — si può passare alla Fase B.")

Checkpoint trovato: 3,240 segmenti già descritti.
Da descrivere: 0
✓ Tutti i segmenti già descritti — si può passare alla Fase B.


In [10]:
%%time
from openai import OpenAI

client = OpenAI()

new_rows = []
n_ok = n_error = 0
total = len(segments_todo)

for i, (_, seg) in enumerate(segments_todo.iterrows()):

    prompt = build_prompt_functional_description(
        testo          = seg['testo'],
        tipo           = seg['tipo'],
        identificatore = seg['identificatore'],
        title_atto     = seg['title_atto'],
        tema           = TEMA_DESCRIZIONE,
    )
    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)
    if status != 'ok':
        print(f"  [{i+1}] ERRORE: {descrizione}")

    new_rows.append({
        'segment_id':              seg['segment_id'],
        'celex':                   seg['celex'],
        'node_id':                 seg['node_id'],
        'tipo':                    seg['tipo'],
        'identificatore':          seg['identificatore'],
        'testo_originale':         seg['testo'],
        'descrizione_funzionale':  descrizione,
        'llm_status':              status,
    })

    if status == 'ok':
        n_ok += 1
    else:
        n_error += 1

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total:
        batch    = pd.DataFrame(new_rows)
        combined = pd.concat([segs_done, batch], ignore_index=True) if not segs_done.empty else batch
        combined.to_csv(SEGMENTS_DESC_FILE, index=False)
        print(f"  [{i+1:>5}/{total}]  {(i+1)/total*100:5.1f}%   ok: {n_ok}   errori: {n_error}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print(f"FASE A — ok: {n_ok:,}   errori: {n_error:,}")
print("=" * 50)


FASE A — ok: 0   errori: 0
CPU times: total: 359 ms
Wall time: 381 ms


## 4b. Fase A2 — Assegnazione Livelli Lamfalussy per Segmento (LLM 2)

Partendo dall'output di Fase A (`segments_descriptions.csv`), per ogni segmento
l'LLM assegna una distribuzione percentuale sui 4 livelli Lamfalussy.

La **descrizione del livello di astrazione** prodotta in Fase A viene fornita come
contesto: l'LLM non deve più dedurre la posizione gerarchica dal testo,
deve solo mappare quella descrizione sulla tassonomia Lamfalussy standard.

**Output**: `segments_lamfalussy.csv` — una riga per segmento con
`lamf_L1`, `lamf_L2`, `lamf_L3`, `lamf_L4` (somma = 100).

> Gira su **tutti** i segmenti (considerando + articoli + allegati).
> Fase C2 filtrerà ai soli articoli per il calcolo dell'entropia.

In [11]:
# ── Costanti Lamfalussy (usate in A2, C2, visualizzazioni) ────────────────────
import json
LAMFALUSSY_LEVELS = [
    {
        'key':         'L1',
        'name':        'L1 — Primary principles and norms',
        'description': (
            'Fundamental principles, enabling acts, constitutional norms '
            'and provisions that establish the general legal framework '
            'and delegate secondary rule-making powers.'
        ),
    },
    {
        'key':         'L2',
        'name':        'L2 — Transposition and codification norms',
        'description': (
            'Legislative decrees, codes and ordinary laws that transpose '
            'EU directives or codify sectoral regulation, defining '
            'rights, obligations and general procedures.'
        ),
    },
    {
        'key':         'L3',
        'name':        'L3 — Implementing and regulatory norms',
        'description': (
            'Regulations, implementing decrees, guidelines and soft law acts '
            'that specify the operational modalities of application '
            'of primary norms.'
        ),
    },
    {
        'key':         'L4',
        'name':        'L4 — Procedural and operational norms',
        'description': (
            'Detailed technical-procedural provisions: numerical thresholds, '
            'deadlines, forms, publication obligations, formal requirements '
            'and enforcement mechanisms.'
        ),
    },
]

LAMF_KEYS = [l['key'] for l in LAMFALUSSY_LEVELS]          # ['L1','L2','L3','L4']
LAMF_COLS = [f'lamf_{k}' for k in LAMF_KEYS]               # ['lamf_L1',...]


def build_prompt_lamfalussy_assignment(testo, identificatore, tipo):
    levels_text = '\n'.join([
        f"  {l['key']}: {l['name']} — {l['description']}"
        for l in LAMFALUSSY_LEVELS
    ])
    return f"""You are an expert in Italian administrative and procurement law.

Distribute its content as a percentage across these four hierarchical levels
of the Italian regulatory system for public procurement:

{levels_text}

Rules:
- Percentages must sum to exactly 100.
- Assign 0 to levels not present in the segment.
- Base your judgment on the normative function, not the subject matter.
- For each level with percentage > 0, identify 1-3 short excerpts from the 
  segment text that justify the assignment and explain why in one sentence.

Segment ({tipo} {identificatore}):
{testo}

Reply ONLY with valid JSON IN ENGLISH (no backticks):
{{
  "L1": X, "L2": X, "L3": X, "L4": X,
  "evidence": [
    {{"testo": "exact short quote from segment", "layer": "L1", "motivo": "one sentence explanation"}},
    ...
  ]
}}"""


print("Costanti Lamfalussy e funzioni Fase A2 definite.")
print(f"Livelli: {LAMF_KEYS}")


Costanti Lamfalussy e funzioni Fase A2 definite.
Livelli: ['L1', 'L2', 'L3', 'L4']


In [12]:
%%time

def parse_percentage_response(response_text, keys):
    import json, re
    text = response_text.strip()
    text = re.sub(r'^```[a-z]*\n?', '', text)
    text = re.sub(r'\n?```$', '', text)
    try:
        data = json.loads(text)
        result = {}
        for k in keys:
            val = float(data.get(k, 0))
            result[k] = max(0.0, min(100.0, val))
        total = sum(result.values())
        if total > 0 and abs(total - 100) > 1:
            result = {k: v / total * 100 for k, v in result.items()}
        result['_evidence'] = data.get('evidence', [])
        return result
    except Exception:
        return None

# ── Gestione checkpoint ────────────────────────────────────────────────────────
# Input: articles_df (900 articoli interi, non paragrafi)
done_lamf_ids = set()
if os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
    lamf_segs_done = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
    done_lamf_ids  = set(lamf_segs_done['segment_id'])
    print(f"Checkpoint: {len(done_lamf_ids):,} già classificati.")
else:
    lamf_segs_done = pd.DataFrame()

articles_df = segments_df[segments_df['tipo'] == 'articolo'].copy()

segs_a_todo = articles_df[~articles_df['segment_id'].isin(done_lamf_ids)].copy()
print(f"Articoli da classificare: {len(segs_a_todo):,}")

new_lamf_rows = []
n_ok_a2 = n_error_a2 = 0
total_a2 = len(segs_a_todo)

for i, (_, seg) in enumerate(segs_a_todo.iterrows()):

    prompt = build_prompt_lamfalussy_assignment(
        testo          = seg['testo'],
        identificatore = seg['identificatore'],
        tipo           = seg['tipo'],
    )
    response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_A2)

    distribution = None
    if status == 'ok':
        distribution = parse_percentage_response(response_text, LAMF_KEYS)
        if distribution is None:
            status = 'parse_error'

    row = {
        'segment_id':    seg['segment_id'],
        'celex':         seg['celex'],
        'node_id':       seg['node_id'],
        'tipo':          seg['tipo'],
        'identificatore': seg['identificatore'],
        'llm_status':    status,
        'evidence':      json.dumps(distribution.get('_evidence', []),
                                    ensure_ascii=False) if distribution else '[]',
    }
    for col, key in zip(LAMF_COLS, LAMF_KEYS):
        row[col] = round(distribution[key], 2) if distribution else 0.0

    if distribution:
        n_ok_a2 += 1
    else:
        n_error_a2 += 1

    new_lamf_rows.append(row)

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total_a2:
        batch    = pd.DataFrame(new_lamf_rows)
        combined = pd.concat([lamf_segs_done, batch], ignore_index=True) if not lamf_segs_done.empty else batch
        combined.to_csv(SEGMENTS_LAMF_CKPT_FILE, index=False)
        print(f"  [{i+1:>5}/{total_a2}]  {(i+1)/total_a2*100:5.1f}%   ok: {n_ok_a2}   errori: {n_error_a2}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print('=' * 50)
print(f"FASE A2 — ok: {n_ok_a2:,}   errori: {n_error_a2:,}")
print('=' * 50)


Checkpoint: 906 già classificati.
Articoli da classificare: 0

FASE A2 — ok: 0   errori: 0
CPU times: total: 15.6 ms
Wall time: 23.8 ms


In [13]:
# Salva output Fase A2
segments_lamf_final = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
segments_lamf_final.to_csv(SEGMENTS_LAMF_FILE, index=False)

print(f"Salvato: {SEGMENTS_LAMF_FILE}")
print(f"Segmenti totali: {len(segments_lamf_final):,}  |  Colonne Lamfalussy: {LAMF_COLS}")
print()

ok_mask = segments_lamf_final['llm_status'] == 'ok'
print("Distribuzione media % per livello Lamfalussy (tutti i segmenti ok):")
for col, key in zip(LAMF_COLS, LAMF_KEYS):
    level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
    mean_pct = segments_lamf_final.loc[ok_mask, col].mean()
    bar      = '█' * int(mean_pct / 2)
    print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")

print()
# Breakdown per tipo di segmento
print("Distribuzione per tipo di segmento:")
for tipo in ['considerando', 'articolo', 'allegato']:
    n = ok_mask & (segments_lamf_final['tipo'] == tipo)
    if n.sum() > 0:
        print(f"  {tipo:>12}: {n.sum():>4} segmenti")


Salvato: ..\data\output\appalti_it\segments_lamfalussy.csv
Segmenti totali: 906  |  Colonne Lamfalussy: ['lamf_L1', 'lamf_L2', 'lamf_L3', 'lamf_L4']

Distribuzione media % per livello Lamfalussy (tutti i segmenti ok):
  L1 — Primary principles and norms.............   7.9%  ███
  L2 — Transposition and codification norms.....  47.0%  ███████████████████████
  L3 — Implementing and regulatory norms........  13.6%  ██████
  L4 — Procedural and operational norms.........  31.5%  ███████████████

Distribuzione per tipo di segmento:
      articolo:  892 segmenti


## 5. Fase B — Embedding e Clustering → Layer Emergenti

Le descrizioni funzionali vengono embeddate con `all-mpnet-base-v2`, ridotte con
UMAP e clusterizzate con HDBSCAN. Ogni cluster che emerge è un **livello gerarchico
specifico per questa materia** — quanti ce ne sono lo decide l'algoritmo.

I segmenti assegnati al cluster `-1` (noise) vengono conservati ma esclusi dal naming.

In [14]:
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

# Carica dal checkpoint finale
segs_desc = pd.read_csv(SEGMENTS_DESC_FILE)

segs_valid = segs_desc[
    (segs_desc['llm_status'] == 'ok') &
    segs_desc['descrizione_funzionale'].notna() &
    (segs_desc['descrizione_funzionale'].str.len() > 10)
].copy()

print(f"Segmenti con descrizione valida: {len(segs_valid):,} / {len(segs_desc):,}")
print()

# ── Embedding con cache ───────────────────────────────────────────────────────
embeddings_file = os.path.join(output_path, 'embeddings.npy')

if os.path.exists(embeddings_file):
    embeddings = np.load(embeddings_file)
    print(f"Embeddings caricati da cache: {embeddings.shape}")
else:
    print(f"Caricamento modello embedding: {EMBEDDING_MODEL} ...")
    encoder = SentenceTransformer(EMBEDDING_MODEL)
    print("Calcolo embeddings...")
    embeddings = encoder.encode(
        segs_valid['descrizione_funzionale'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    print(f"Embeddings shape: {embeddings.shape}")
    np.save(embeddings_file, embeddings)
    print(f"Embeddings salvati: {embeddings_file}")

Segmenti con descrizione valida: 3,240 / 3,240

Embeddings caricati da cache: (3240, 768)


In [15]:
import numpy as np

embeddings_file = os.path.join(output_path, 'embeddings.npy')
np.save(embeddings_file, embeddings)
print(f"Embeddings salvati: {embeddings_file}")

Embeddings salvati: ..\data\output\appalti_it\embeddings.npy


In [16]:
print(f"UMAP: {embeddings.shape[1]}d → {UMAP_N_COMPONENTS}d ...")

reducer = umap.UMAP(
    n_components = UMAP_N_COMPONENTS,
    n_neighbors  = UMAP_N_NEIGHBORS,
    min_dist     = UMAP_MIN_DIST,
    metric       = 'cosine',
    random_state = 42,
    low_memory   = False,
)
embeddings_reduced = reducer.fit_transform(embeddings)
print(f"Shape ridotta: {embeddings_reduced.shape}")

UMAP: 768d → 10d ...


c:\Users\claud\Documents\GitHub\eu-law-network-viz\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Shape ridotta: (3240, 10)


In [17]:
print(f"HDBSCAN (min_cluster={HDBSCAN_MIN_CLUSTER}, min_samples={HDBSCAN_MIN_SAMPLES}) ...")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size         = HDBSCAN_MIN_CLUSTER,
    min_samples              = HDBSCAN_MIN_SAMPLES,
    cluster_selection_method = 'eom',
    prediction_data          = True,
)
cluster_labels = clusterer.fit_predict(embeddings_reduced)

segs_valid = segs_valid.copy()
segs_valid['cluster_id'] = cluster_labels
if hasattr(clusterer, 'probabilities_'):
    segs_valid['cluster_prob'] = clusterer.probabilities_

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()

print()
print("=" * 50)
print("RISULTATO CLUSTERING")
print("=" * 50)
print(f"  Layer trovati:          {n_clusters}")
print(f"  Segmenti noise (-1):    {n_noise:,}  ({n_noise/len(cluster_labels)*100:.1f}%)")
print()

cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print("Distribuzione segmenti per cluster:")
for cid, cnt in cluster_counts.items():
    tag = 'NOISE' if cid == -1 else f'cluster_{cid}'
    bar = '█' * min(40, int(cnt / cluster_counts.max() * 40))
    print(f"  {tag:>12}: {cnt:>5}  {bar}")

HDBSCAN (min_cluster=80, min_samples=20) ...

RISULTATO CLUSTERING
  Layer trovati:          12
  Segmenti noise (-1):    18  (0.6%)

Distribuzione segmenti per cluster:
         NOISE:    18  █
     cluster_0:   108  ██████
     cluster_1:   672  ████████████████████████████████████████
     cluster_2:   146  ████████
     cluster_3:   439  ██████████████████████████
     cluster_4:   175  ██████████
     cluster_5:   185  ███████████
     cluster_6:   103  ██████
     cluster_7:   628  █████████████████████████████████████
     cluster_8:   269  ████████████████
     cluster_9:   248  ██████████████
    cluster_10:    87  █████
    cluster_11:   162  █████████


## 6. Fase B.2 — Ordinamento Gerarchico e Naming dei Layer (LLM 2)

**Naming**: LLM 2 riceve le 10 descrizioni più rappresentative di ogni cluster e assegna un nome e una descrizione al layer,
ed infine li ordina gerarchicamente.

In [18]:
from openai import OpenAI

client = OpenAI()

# ── Funzioni ──────────────────────────────────────────────────────────────────

def normalize_cluster_id(x):
    return int(str(x).replace('cluster_', '').strip())

def call_llm(client, prompt, max_tokens):
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=max_tokens,
        )
        return response.choices[0].message.content.strip(), 'ok'
    except Exception as e:
        print(f"  Errore LLM: {e}")
        return '', 'error'


def get_representative_descriptions(segs_df, cluster_id, n=N_REPR_DESCRIPTIONS):
    subset = segs_df[segs_df['cluster_id'] == cluster_id].copy()
    if 'cluster_prob' in subset.columns:
        subset = subset.nlargest(n, 'cluster_prob')
    else:
        subset = subset.sample(min(n, len(subset)), random_state=42)
    return subset['descrizione_funzionale'].tolist()


def build_prompt_layer_naming(descriptions):
    desc_list = '\n'.join([f"  {i+1}. {d}" for i, d in enumerate(descriptions)])
    return f"""You are an expert in Italian law, 
    with knowledge of EU legislation and its transposition into Italian law.

Below are functional descriptions of legal segments that cluster together 
in a corpus analysis. They share the same normative role in the regulatory 
hierarchy.

Identify what hierarchical function unites them and give this cluster:
- a SHORT name (3-6 words, functional not thematic)
- a brief description (2-3 sentences) of the normative role

GOOD name examples:
- "Empowerment and delegation clauses"
- "Justificatory and context-setting recitals"
- "Commencement and temporal provisions"
- "Procedural safeguards and consultation requirements"
- "Scope-defining and definitional provisions"

BAD name examples (too thematic, too long):
- "Foundational purposes and cooperative architecture of EU FDI screening"
- "Confidentiality rules in foreign subsidy investigations"

Representative segments:
{desc_list}

Reply ONLY in this JSON format (no backticks):
{{"nome": "Layer Name", "descrizione": "Description in 2-3 sentences."}}"""


def build_prompt_layer_ranking(layer_records):
    layers_text = '\n'.join([
        f"  cluster_{r['cluster_id']}: {r['layer_name']} — {r['layer_description'][:120]}"
        for r in layer_records
    ])
    return f"""You are an expert in Italian law, 
    with knowledge of EU legislation and its transposition into Italian law.

Below are functional layers found in a corpus of EU legal acts.
Rank them from most foundational (1 = constitutional basis, enabling norms, 
definitions) to most technical and operational (n = filing rules, timing, 
cross-references).

Layers:
{layers_text}

Reply ONLY with a JSON array of cluster_ids in order from most foundational 
to most technical (no other text, no backticks):
[cluster_id_1, cluster_id_2, ..., cluster_id_n]"""


def build_prompt_layer_merge(layer_records):
    layers_text = '\n'.join([
        f"  cluster_{r['cluster_id']}: \"{r['layer_name']}\" — {r['layer_description'][:150]}"
        for r in layer_records
    ])
    return f"""You are an expert in legal taxonomy.

Below are clusters from a legal text analysis. Your task is to identify which 
clusters should be merged because they represent the SAME normative function.

MANDATORY merge triggers (you MUST merge if any of these apply):
1. Two or more clusters have identical or near-identical names.
2. Two or more clusters have descriptions that describe the same hierarchical 
   position with different words (e.g. "operational implementing rules" and 
   "implementing procedural rules" are the same function).
3. The only difference between two clusters is stylistic, not functional.

Do NOT merge:
- Clusters at genuinely different hierarchical levels (e.g. foundational vs operational).
- Clusters with distinct structural functions (e.g. exclusion/exemption clauses 
  vs general procedural rules).

Clusters:
{layers_text}

IMPORTANT: clusters with identical names MUST be merged.
IMPORTANT: only include groups with 2 or more clusters — do not list single clusters.

Reply ONLY with a JSON array of merge groups (lists of cluster_ids to merge).
Clusters not in any group stay as-is.
Example: [[2, 4, 5], [6, 7]] means merge 2+4+5 together and 6+7 together.
If truly nothing to merge: []
(no backticks, no other text)"""


# ── Fase B.2 — Naming ─────────────────────────────────────────────────────────

if os.path.exists(LAYER_MAPPING_FILE):
    layer_mapping_df = pd.read_csv(LAYER_MAPPING_FILE)
    layer_records = layer_mapping_df.to_dict('records')
    print(f"Checkpoint trovato: {len(layer_records)} layer già nominati e ordinati.")
else:
    valid_cluster_ids = sorted([
    cid for cid, count in {
        cid: int((segs_valid['cluster_id'] == cid).sum())
        for cid in set(cluster_labels) if cid != -1
    }.items()
    if count > 0
])

    original_counts = {
        cid: int((segs_valid['cluster_id'] == cid).sum())
        for cid in valid_cluster_ids
    }

    print("DEBUG original_counts:", original_counts)

    layer_records = []
    for cluster_id in valid_cluster_ids:
        repr_descs  = get_representative_descriptions(segs_valid, cluster_id)
        prompt      = build_prompt_layer_naming(repr_descs)
        response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_B2)

        nome        = f'Layer_{cluster_id}'
        descrizione = ''
        if status == 'ok':
            try:
                clean       = response_text.replace('```json', '').replace('```', '').strip()
                parsed      = json.loads(clean)
                nome        = parsed.get('nome', nome)
                descrizione = parsed.get('descrizione', '')
            except json.JSONDecodeError:
                nome = response_text[:80].strip()
                print(f"  cluster_{cluster_id}: JSON non valido, uso raw text come nome")

        layer_records.append({
            'cluster_id':        cluster_id,
            'layer_rank':        None,
            'layer_name':        nome,
            'layer_description': descrizione,
            'n_segments':        original_counts[cluster_id],
            'repr_descriptions': json.dumps(repr_descs, ensure_ascii=False),
            'llm_status':        status,
        })
        print(f"  cluster_{cluster_id:>3} → '{nome}'")
        time.sleep(LLM_DELAY_SECONDS)


    # ── Fase B.2b — Merge cluster simili (LLM) ───────────────────────────────────

    print("\nValutazione merge cluster simili via LLM...")
    prompt_merge         = build_prompt_layer_merge(layer_records)
    response_merge, status_merge = call_llm(client, prompt_merge, 300)

    if status_merge == 'ok':
        try:
            clean_merge  = response_merge.replace('```json', '').replace('```', '').strip()
            merge_groups = json.loads(clean_merge)
            merge_groups = [g for g in merge_groups if len(g) >= 2]

            if merge_groups:
                merge_map = {}
                canonicals = set()

                # FIX 1: normalizzazione robusta + canonicals
                for group in merge_groups:
                    group     = [normalize_cluster_id(x) for x in group]
                    canonical = group[0]
                    canonicals.add(canonical)
                    for cid in group:
                        merge_map[cid] = canonical

                print("DEBUG merge_groups:", merge_groups)
                print("DEBUG merge_map:", merge_map)

                segs_valid['cluster_id'] = segs_valid['cluster_id'].apply(
                    lambda x: merge_map.get(int(x), int(x))
                )

                non_merged = [
                    dict(r) for r in layer_records
                    if int(r['cluster_id']) not in merge_map
                ]

                merged_records = []
                for group in merge_groups:
                    group     = [normalize_cluster_id(x) for x in group]
                    canonical = group[0]
                    parts     = [r for r in layer_records if int(r['cluster_id']) in group]

                    # FIX 3: somma sicura + debug
                    counts = [original_counts.get(int(p['cluster_id']), 0) for p in parts]
                    print(f"DEBUG merging {group} → counts {counts}")

                    n_tot = sum(counts)

                    merged_records.append({
                        'cluster_id':        canonical,
                        'layer_rank':        None,
                        'layer_name':        parts[0]['layer_name'],
                        'layer_description': parts[0]['layer_description'],
                        'n_segments':        n_tot,
                        'repr_descriptions': parts[0]['repr_descriptions'],
                        'llm_status':        'merged',
                        'merged_from':       str([int(p['cluster_id']) for p in parts]),
                    })

                    print(f"  Merge: cluster_{'+'.join(str(x) for x in group)} "
                          f"→ '{parts[0]['layer_name']}'  (n={n_tot})")

                layer_records = non_merged + merged_records

                print(f"  Layer dopo merge: {len(layer_records)}")
                print(f"  Totale segmenti: {sum(r['n_segments'] for r in layer_records)}"
                      f" / {len(segs_valid)}")

            else:
                print("  Nessun merge suggerito — tutti i cluster sono distinti.")

        except (json.JSONDecodeError, TypeError, ValueError) as e:
            print(f"  Merge LLM non valido ({e}) — nessun merge applicato.")
    else:
        print("  Merge LLM fallito — nessun merge applicato.")


    # ── Fase B.3 — Ranking LLM ───────────────────────────────────────────────────

    print("\nOrdinamento gerarchico via LLM...")
    prompt_rank           = build_prompt_layer_ranking(layer_records)
    response_rank, status_rank = call_llm(client, prompt_rank, 200)

    if status_rank == 'ok':
        try:
            clean       = response_rank.replace('```json', '').replace('```', '').strip()
            ordered_ids = [normalize_cluster_id(x) for x in json.loads(clean)]
            llm_rank    = {cid: rank + 1 for rank, cid in enumerate(ordered_ids)}
            for i, r in enumerate([r for r in layer_records
                                    if int(r['cluster_id']) not in llm_rank]):
                llm_rank[int(r['cluster_id'])] = len(ordered_ids) + i + 1
                print(f"  Warning: cluster_{r['cluster_id']} non nel ranking, appeso in fondo")
        except (json.JSONDecodeError, TypeError) as e:
            print(f"  Ranking LLM non valido ({e}) — fallback su n_segments")
            llm_rank = {int(r['cluster_id']): rank + 1
                        for rank, r in enumerate(
                            sorted(layer_records, key=lambda r: r['n_segments'], reverse=True))}
    else:
        print("  Ranking LLM fallito — fallback su n_segments")
        llm_rank = {int(r['cluster_id']): rank + 1
                    for rank, r in enumerate(
                        sorted(layer_records, key=lambda r: r['n_segments'], reverse=True))}

    for r in layer_records:
        r['layer_rank'] = llm_rank[int(r['cluster_id'])]


# ── Salvataggio e stampa ──────────────────────────────────────────────────────

layer_mapping_df = pd.DataFrame(layer_records).sort_values('layer_rank')
layer_mapping_df.to_csv(LAYER_MAPPING_FILE, index=False)

print()
print("=" * 60)
print("LAYER TROVATI")
print("=" * 60)
for _, row in layer_mapping_df.iterrows():
    print(f"  [{row['layer_rank']}] {row['layer_name']}")
    print(f"      {str(row['layer_description'])}")
    print(f"      N segmenti: {row['n_segments']:,}")
    print()

Checkpoint trovato: 6 layer già nominati e ordinati.

LAYER TROVATI
  [1] Foundational and structural provisions
      This cluster groups provisions that sit above ordinary operational rules and organize the legal framework at a high level. They set overarching principles, define the scope or interpretive approach of the code, and allocate general responsibilities or default regimes, often leaving concrete implementation to subordinate or later rules.
      N segmenti: 175

  [2] Scope and exclusion clauses
      These provisions delimit the material or personal scope of a legal instrument by carving out specific cases, categories, or situations from the general regime. They function as technical boundary-setting rules, often through explicit exceptions, cross-references, and conditions that determine when the main framework does not apply.
      N segmenti: 185

  [3] Exception and derogation clauses
      These provisions operate at a technical level to carve out specific cases, con

## 7. Fase C — Distribuzione Percentuale per Articolo (LLM 3)

Con i layer noti, per ogni **articolo** di ogni atto l'LLM produce una distribuzione
percentuale del contenuto tra i layer emersi.

Il risultato per ogni atto è la matrice **articoli × layer** (valori = %) che alimenta
la heatmap nell'applicazione.

> **Perché solo gli articoli?** I considerando hanno funzione giustificativa e retorica:
> spesso coprono più livelli intenzionalmente per costruire l'argomentazione legale.
> La varianza dei considerando riflette struttura retorica, non patologia.
> Gli articoli hanno funzione prescrittiva — la loro ibridità è il segnale diagnostico.

In [19]:
# Ricarica layer mapping (può essere eseguita anche senza rieseguire B)
layer_mapping_df = pd.read_csv(LAYER_MAPPING_FILE)
layer_list = [
    {
        'rank':        int(row['layer_rank']),
        'name':        row['layer_name'],
        'description': row['layer_description'],
    }
    for _, row in layer_mapping_df.sort_values('layer_rank').iterrows()
]
layer_names = [l['name'] for l in layer_list]

# Colonne CSV safe (senza spazi/slash)
def to_col(name):
    return 'pct__' + name.replace(' ', '_').replace('/', '_')[:50]

pct_cols     = [to_col(n) for n in layer_names]
col_to_layer = {to_col(n): n for n in layer_names}

# Solo gli articoli
articles_df = segments_df[segments_df['tipo'] == 'articolo'].copy()

print(f"Layer trovati: {len(layer_list)}")
for l in layer_list:
    print(f"  [{l['rank']}] {l['name']}")
print()
print(f"Articoli da classificare: {len(articles_df):,}")
print(f"Atti coinvolti:           {articles_df['celex'].nunique():,}")

Layer trovati: 6
  [1] Foundational and structural provisions
  [2] Scope and exclusion clauses
  [3] Exception and derogation clauses
  [4] Implementation and procedural rules
  [5] Cross-reference and incorporation clauses
  [6] Technical amending provisions

Articoli da classificare: 906
Atti coinvolti:           21


In [20]:
def build_prompt_percentage_distribution(testo, identificatore, layer_list, tema):
    layers_text = '\n'.join([
        f"  {l['name']}: {l['description']}"
        for l in layer_list
    ])
    return f"""You are an expert in legal analysis specializing in: {tema}

Read this article and:
1. Distribute its content as a percentage across the normative layers below.
2. For each layer with percentage > 0, quote 1-3 short excerpts from the text
   that justify the assignment and explain why in one sentence.

Layers:
{layers_text}

Rules:
- Percentages must sum to exactly 100.
- Assign 0 to layers not present.
- Quotes must be exact substrings of the article text.

Article {identificatore}:
{testo}

Reply ONLY with valid JSON (no backticks):
{{
  "Layer Name 1": X,
  "Layer Name 2": X,
  ...,
  "evidence": [
    {{"testo": "exact quote", "layer": "Layer Name", "motivo": "one sentence explanation"}},
    ...
  ]
}}"""


def parse_percentage_response(response_text, layer_names):
    """
    Parsa la risposta JSON e normalizza a somma 100.
    Restituisce None se il parsing fallisce.
    """
    try:
        clean  = response_text.replace('```json', '').replace('```', '').strip()
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        return None

    values = {name: float(parsed.get(name, 0)) for name in layer_names}
    total  = sum(values.values())
    if total <= 0:
        return None
    if abs(total - 100) > 5:   # normalizza se la somma si discosta
        values = {k: v / total * 100 for k, v in values.items()}
    return values


print("Funzioni Fase C definite.")

Funzioni Fase C definite.


In [21]:
%%time
import json as _json
# ── Gestione checkpoint ────────────────────────────────────────────────────────
if os.path.exists(HEATMAP_CKPT_FILE):
    heatmap_done = pd.read_csv(HEATMAP_CKPT_FILE)
    done_seg_ids = set(heatmap_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_seg_ids):,} articoli già classificati.")
else:
    heatmap_done = pd.DataFrame()
    done_seg_ids = set()
    print("Nessun checkpoint heatmap — si parte da zero.")

articles_todo = articles_df[~articles_df['segment_id'].isin(done_seg_ids)].copy()
print(f"Articoli da classificare: {len(articles_todo):,}")
print()

new_rows = []
n_ok_c = n_error_c = 0
total_c = len(articles_todo)

for i, (_, art) in enumerate(articles_todo.iterrows()):

    prompt = build_prompt_percentage_distribution(
        testo          = art['testo'],
        identificatore = art['identificatore'],
        layer_list     = layer_list,
        tema           = TEMA_DESCRIZIONE,
    )
    response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_C)

    distribution = None
    if status == 'ok':
        distribution = parse_percentage_response(response_text, layer_names)
        if distribution is None:
            status = 'parse_error'

    row = {
        'segment_id':  art['segment_id'],
        'celex':       art['celex'],
        'node_id':     art['node_id'],
        'articolo_id': art['identificatore'],
        'llm_status':  status,
        'evidence':    _json.dumps(distribution.get('_evidence', []),
                       ensure_ascii=False) if distribution else '[]',
    }
    for col, name in zip(pct_cols, layer_names):
        row[col] = round(distribution[name], 2) if distribution else 0.0

    if distribution:
        n_ok_c += 1
    else:
        n_error_c += 1

    new_rows.append(row)

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total_c:
        batch    = pd.DataFrame(new_rows)
        combined = pd.concat([heatmap_done, batch], ignore_index=True) if not heatmap_done.empty else batch
        combined.to_csv(HEATMAP_CKPT_FILE, index=False)
        print(f"  [{i+1:>5}/{total_c}]  {(i+1)/total_c*100:5.1f}%   ok: {n_ok_c}   errori: {n_error_c}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print(f"FASE C — ok: {n_ok_c:,}   errori: {n_error_c:,}")
print("=" * 50)

Checkpoint trovato: 906 articoli già classificati.
Articoli da classificare: 0


FASE C — ok: 0   errori: 0
CPU times: total: 0 ns
Wall time: 4.78 ms


In [22]:
# Salva heatmap finale
heatmap_final = pd.read_csv(HEATMAP_CKPT_FILE)
heatmap_final.rename(columns={'celex': 'id'}).to_csv(NODES_HEATMAP_FILE, index=False)

print(f"Salvato: {NODES_HEATMAP_FILE}")
print(f"Righe: {len(heatmap_final):,}  |  Colonne pct: {pct_cols}")
print()

ok_mask = heatmap_final['llm_status'] == 'ok'
print("Distribuzione media % per layer (articoli ok):")
for col in pct_cols:
    mean_pct = heatmap_final.loc[ok_mask, col].mean()
    bar = '█' * int(mean_pct / 2)
    print(f"  {col_to_layer[col][:45]:.<46} {mean_pct:5.1f}%  {bar}")

Salvato: ..\data\output\appalti_it\nodes_heatmap.csv
Righe: 906  |  Colonne pct: ['pct__Foundational_and_structural_provisions', 'pct__Scope_and_exclusion_clauses', 'pct__Exception_and_derogation_clauses', 'pct__Implementation_and_procedural_rules', 'pct__Cross-reference_and_incorporation_clauses', 'pct__Technical_amending_provisions']

Distribuzione media % per layer (articoli ok):
  Foundational and structural provisions........  14.4%  ███████
  Scope and exclusion clauses...................  10.9%  █████
  Exception and derogation clauses..............  12.6%  ██████
  Implementation and procedural rules...........  35.8%  █████████████████
  Cross-reference and incorporation clauses.....  15.1%  ███████
  Technical amending provisions.................  11.3%  █████


## 7b. Fase C2 — Entropia Lamfalussy per Articolo

Calcola lo **score di ibridità Lamfalussy** partendo dall'output di Fase A2.
Nessuna chiamata LLM — è una trasformazione puramente computazionale.

Per ogni atto produce:
- `hybridity_lamf_score` — entropia media degli articoli su L1–L4 (metrica principale)
- `dominant_lamf` — livello Lamfalussy dominante

**Input**: `segments_lamfalussy.csv` (tutti i segmenti, filtrati agli articoli)

**Output**: `nodes_lamfalussy.csv` (articoli con distribuzione L1–L4, per le visualizzazioni)

In [23]:
# ── Fase C2: entropia Lamfalussy per articolo ────────────────────
if not os.path.exists(SEGMENTS_LAMF_FILE):
    print(f"  {SEGMENTS_LAMF_FILE} non trovato — esegui prima Fase A2 (sezione 4b).")
else:
    segments_lamf = pd.read_csv(SEGMENTS_LAMF_FILE)

    # Filtra ai soli articoli (come Fase C sui layer emersi)
    articles_lamf = segments_lamf[
        (segments_lamf['llm_status'] == 'ok') &
        (segments_lamf['segment_id'].isin(set(articles_df['segment_id'])))
    ].copy()

    # Assicura che le colonne LAMF_COLS esistano
    for col in LAMF_COLS:
        if col not in articles_lamf.columns:
            articles_lamf[col] = 0.0

    # Rinomina come nodes_lamfalussy.csv (compatibilità con le celle di visualizzazione)
    # aggiunge articolo_id per allineamento con heatmap_ok
    articles_lamf['articolo_id'] = articles_lamf['identificatore']
    articles_lamf.to_csv(NODES_LAMFALUSSY_FILE, index=False)

    print(f"Salvato: {NODES_LAMFALUSSY_FILE}")
    print(f"Articoli: {len(articles_lamf):,}  |  Atti: {articles_lamf['celex'].nunique():,}")
    print()
    print("Distribuzione media % per livello Lamfalussy (articoli):")
    for col, key in zip(LAMF_COLS, LAMF_KEYS):
        level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
        mean_pct = articles_lamf[col].mean()
        bar      = '█' * int(mean_pct / 2)
        print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")


Salvato: ..\data\output\appalti_it\nodes_lamfalussy.csv
Articoli: 892  |  Atti: 21

Distribuzione media % per livello Lamfalussy (articoli):
  L1 — Primary principles and norms.............   7.9%  ███
  L2 — Transposition and codification norms.....  47.0%  ███████████████████████
  L3 — Implementing and regulatory norms........  13.6%  ██████
  L4 — Procedural and operational norms.........  31.5%  ███████████████


## 8. Fase D — Score di Ibridità per Atto

Lo score di ibridità misura quanto gli articoli di un atto variano nel loro livello
gerarchico. Un atto **puro** ha tutti gli articoli concentrati sullo stesso layer.
Un atto **ibrido** ha articoli che spaziano su layer molto diversi.

**Metrica: entropia di Shannon normalizzata per articolo**

$$H(a) = -\sum_{l} p_{al} \log_2(p_{al} + \varepsilon)$$

Lo score dell'atto è la **media delle entropie dei propri articoli**.
Zero = tutti gli articoli sono monofunzionali. Uno = distribuzione uniforme su tutti i layer.

In [24]:
def entropy_norm(row, pct_cols):
    """Entropia di Shannon normalizzata (0=puro, 1=uniforme)."""
    eps   = 1e-9
    probs = np.array([float(row.get(c, 0)) for c in pct_cols]) / 100.0
    probs = np.clip(probs, 0, 1)
    s     = probs.sum()
    if s < eps:
        return 0.0
    probs = probs / s
    raw   = -np.sum(probs * np.log2(probs + eps))
    maxH  = math.log2(len(pct_cols)) if len(pct_cols) > 1 else 1.0
    return float(raw / maxH)


def dominant_layer(row, pct_cols, col_to_layer):
    best = max(pct_cols, key=lambda c: float(row.get(c, 0)))
    return col_to_layer.get(best, best)


# ── Articoli classificati correttamente (layer emersi) ────────────────────────
heatmap_ok = heatmap_final[heatmap_final['llm_status'] == 'ok'].copy()

# Entropia su layer emersi — metrica secondaria
heatmap_ok['entropy_layers'] = heatmap_ok.apply(
    lambda r: entropy_norm(r, pct_cols), axis=1
)
heatmap_ok['dominant_layer'] = heatmap_ok.apply(
    lambda r: dominant_layer(r, pct_cols, col_to_layer), axis=1
)

# ── Lamfalussy: definizione colonne (self-contained) ──────────────────────────
# Definite qui per garantire disponibilità anche se la cella 7b non è stata eseguita
_LAMF_KEYS_D = ['L1', 'L2', 'L3', 'L4']
_LAMF_COLS_D = [f'lamf_{k}' for k in _LAMF_KEYS_D]

# Merge entropia Lamfalussy in heatmap_ok ────────────────────────────────────
if os.path.exists(NODES_LAMFALUSSY_FILE):
    lamfalussy_final = pd.read_csv(NODES_LAMFALUSSY_FILE)
    lamf_ok = lamfalussy_final[lamfalussy_final['llm_status'] == 'ok'].copy()

    # Assicura che le colonne Lamfalussy esistano (gestisce run parziali)
    for col in _LAMF_COLS_D:
        if col not in lamf_ok.columns:
            lamf_ok[col] = 0.0

    lamf_ok['lamf_entropy'] = lamf_ok.apply(
        lambda r: entropy_norm(r, _LAMF_COLS_D), axis=1
    )
    lamf_ok['dominant_lamf'] = lamf_ok.apply(
        lambda r: max(_LAMF_COLS_D, key=lambda c: float(r.get(c, 0))).replace('lamf_', ''),
        axis=1
    )

    # Merge per-articolo in heatmap_ok: disponibile per le celle di visualizzazione
    heatmap_ok = heatmap_ok.merge(
        lamf_ok[['segment_id', 'lamf_entropy', 'dominant_lamf'] + _LAMF_COLS_D],
        on='segment_id', how='left'
    )
    heatmap_ok['lamf_entropy'] = heatmap_ok['lamf_entropy'].fillna(0.0)
    lamf_available = True
    print(f"Lamfalussy mergato in heatmap_ok: {lamf_ok['lamf_entropy'].notna().sum():,} articoli")
else:
    heatmap_ok['lamf_entropy']  = heatmap_ok['entropy_layers']   # fallback
    heatmap_ok['dominant_lamf'] = heatmap_ok['dominant_layer']
    lamf_available = False
    print(f"  ⚠ {NODES_LAMFALUSSY_FILE} non trovato — uso layer emersi come fallback per lamf_entropy")

# ── Aggregazione per atto ──────────────────────────────────────────────────────
def agg_atto(group):
    return pd.Series({
        # ── primario: Lamfalussy (o fallback su layer emersi) ─────────────────
        'hybridity_score':           group['lamf_entropy'].mean(),
        'hybridity_std':             group['lamf_entropy'].std(),
        'hybridity_max':             group['lamf_entropy'].max(),
        'most_hybrid_article':       (group.nlargest(1, 'lamf_entropy')['articolo_id'].iloc[0]
                                       if len(group) > 0 else ''),
        'dominant_lamf':             (group['dominant_lamf'].mode().iloc[0]
                                       if len(group) > 0 else ''),
        # ── secondario: layer emersi ──────────────────────────────────────────
        'hybridity_layers_score':    group['entropy_layers'].mean(),
        'hybridity_layers_std':      group['entropy_layers'].std(),
        'hybridity_layers_max':      group['entropy_layers'].max(),
        'dominant_layer':            (group['dominant_layer'].mode().iloc[0]
                                       if len(group) > 0 else ''),
        'dominant_layer_pct':        (group['dominant_layer'].value_counts().iloc[0] / len(group) * 100
                                       if len(group) > 0 else 0.0),
        'n_articles':                len(group),
    })

hybridity_df = heatmap_ok.groupby('celex').apply(agg_atto).reset_index()

# Aggiunge metadati dal nodo originale
meta_cols  = [c for c in ['Id', 'Label', 'title', 'LegalType', 'Year', 'PipelineLevel']
              if c in nodes.columns]
nodes_meta = nodes[meta_cols].copy()
nodes_meta = nodes_meta.rename(columns={'Label': 'celex'}) if 'Label' in nodes_meta.columns else nodes_meta

hybridity_df = hybridity_df.merge(nodes_meta, on='celex', how='left')
hybridity_df = hybridity_df.drop_duplicates(subset=['celex'], keep='first')
hybridity_df = hybridity_df.sort_values('hybridity_score', ascending=False)
hybridity_df.rename(columns={'celex': 'id'}).to_csv(NODES_HYBRIDITY_FILE, index=False)

print(f"Salvato: {NODES_HYBRIDITY_FILE}")
print(f"Atti analizzati: {len(hybridity_df):,}")
print()
score_label = "Lamfalussy" if lamf_available else "Layer emersi (fallback)"
desc = hybridity_df['hybridity_score'].describe()
print(f"Statistiche hybridity_score ({score_label}):")
print(f"  Media:   {desc['mean']:.4f}")
print(f"  Mediana: {desc['50%']:.4f}")
print(f"  Max:     {desc['max']:.4f}")
print(f"  Std:     {desc['std']:.4f}")
print()
print("Top 5 atti più ibridi:")
for _, row in hybridity_df.head(5).iterrows():
    celex_label = row.get('Label', row.get('celex', ''))
    print(f"  {celex_label:<20}  score={row['hybridity_score']:.3f}  "
          f"dominant={row['dominant_lamf']}  "
          f"(layers={row['hybridity_layers_score']:.3f})")


Lamfalussy mergato in heatmap_ok: 892 articoli
Salvato: ..\data\output\appalti_it\nodes_hybridity.csv
Atti analizzati: 21

Statistiche hybridity_score (Lamfalussy):
  Media:   0.5414
  Mediana: 0.5694
  Max:     0.7812
  Std:     0.1558

Top 5 atti più ibridi:
  l_55_2019             score=0.781  dominant=L2  (layers=0.764)
  l_120_2020            score=0.757  dominant=L2  (layers=0.860)
  dlgs_218_2012         score=0.713  dominant=L2  (layers=0.761)
  l_108_2021            score=0.700  dominant=L2  (layers=0.791)
  l_114_2014            score=0.694  dominant=L2  (layers=0.819)


## 9. Diagnostica

In [25]:
print("=" * 60)
print("LAYER EMERSI")
print("=" * 60)
for _, row in layer_mapping_df.iterrows():
    n    = row['n_segments']
    pct  = n / len(segs_valid) * 100
    bar  = '█' * int(pct)
    print(f"[{row['layer_rank']:>2}] {row['layer_name']}")
    print(f"     Segmenti: {n:,}  ({pct:.1f}%)  {bar}")
    print(f"     {str(row['layer_description'])[:110]}")
    print()

LAYER EMERSI
[ 1] Foundational and structural provisions
     Segmenti: 175  (5.4%)  █████
     This cluster groups provisions that sit above ordinary operational rules and organize the legal framework at a

[ 2] Scope and exclusion clauses
     Segmenti: 185  (5.7%)  █████
     These provisions delimit the material or personal scope of a legal instrument by carving out specific cases, c

[ 3] Exception and derogation clauses
     Segmenti: 103  (3.2%)  ███
     These provisions operate at a technical level to carve out specific cases, conditions, or categories from an o

[ 4] Implementation and procedural rules
     Segmenti: 2,505  (77.3%)  █████████████████████████████████████████████████████████████████████████████
     These provisions operate at a low, concrete level in the hierarchy, translating higher-level principles into d

[ 5] Cross-reference and incorporation clauses
     Segmenti: 108  (3.3%)  ███
     These provisions do not create autonomous substantive rules; they dire

In [26]:
print("=" * 60)
print("TOP 10 ATTI PIÙ IBRIDI")
print("=" * 60)
print()
for _, row in hybridity_df.head(10).iterrows():
    celex_label = row.get('Label', row.get('celex', ''))
    print(f"  {row['hybridity_score']:.4f}  {celex_label}")
    print(f"           Dominant Lamfalussy: {row.get('dominant_lamf','-')}  |  Layer emerso: {row.get('dominant_layer','-')} ({row.get('dominant_layer_pct',0):.0f}%)")
    print(f"           Art: {row['n_articles']}  |  Più ibrido: {row['most_hybrid_article']}")
    print(f"           {str(row.get('title',''))[:70]}")
    print()

TOP 10 ATTI PIÙ IBRIDI

  0.7812  l_55_2019
           Dominant Lamfalussy: L2  |  Layer emerso: Implementation and procedural rules (57%)
           Art: 21  |  Più ibrido: 4 -sexies
           LEGGE



		



		 	



			14

			giugno

			2019, n. 55 Conversione in

  0.7566  l_120_2020
           Dominant Lamfalussy: L2  |  Layer emerso: Implementation and procedural rules (55%)
           Art: 38  |  Più ibrido: 2 - bis
           LEGGE



		



		 	



			11

			settembre

			2020, n. 120 Conversion

  0.7134  dlgs_218_2012
           Dominant Lamfalussy: L2  |  Layer emerso: Cross-reference and incorporation clauses (43%)
           Art: 7  |  Più ibrido: 3
           DECRETO LEGISLATIVO



		



		 	



			15

			novembre

			2012, n. 2

  0.7002  l_108_2021
           Dominant Lamfalussy: L2  |  Layer emerso: Implementation and procedural rules (47%)
           Art: 45  |  Più ibrido: 36
           LEGGE



		



		 	



			29

			luglio

			2021, n. 108 Conversione i

  0.6936  

## 10. Riepilogo Output

Verifica che tutti i file di output siano stati prodotti correttamente.

In [27]:
output_files = {
    'segments_descriptions.csv': SEGMENTS_DESC_FILE,
    'layer_mapping.csv':         LAYER_MAPPING_FILE,
    'nodes_heatmap.csv':         NODES_HEATMAP_FILE,
    'nodes_hybridity.csv':       NODES_HYBRIDITY_FILE,
    'segments_lamfalussy.csv':    SEGMENTS_LAMF_FILE,
    'nodes_lamfalussy.csv':       NODES_LAMFALUSSY_FILE,
}

print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

all_ok = True
for name, path in output_files.items():
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        df_tmp  = pd.read_csv(path)
        print(f"  ✓ {name}")
        print(f"    Righe: {len(df_tmp):,}  |  Dim: {size_kb:.1f} KB")
        print(f"    Colonne: {list(df_tmp.columns)[:6]}{'...' if len(df_tmp.columns) > 6 else ''}")
    else:
        print(f"  ✗ {name} — FILE MANCANTE")
        all_ok = False
    print()

if all_ok:
    print("✓ Pipeline 04 completata.")
else:
    print("  Alcuni file mancano — rieseguire le celle corrispondenti.")

OUTPUT FILES
  ✓ segments_descriptions.csv
    Righe: 3,240  |  Dim: 6341.7 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'tipo', 'identificatore', 'testo_originale']...

  ✓ layer_mapping.csv
    Righe: 6  |  Dim: 24.0 KB
    Colonne: ['cluster_id', 'layer_rank', 'layer_name', 'layer_description', 'n_segments', 'repr_descriptions']...

  ✓ nodes_heatmap.csv
    Righe: 906  |  Dim: 75.8 KB
    Colonne: ['segment_id', 'id', 'node_id', 'articolo_id', 'llm_status', 'pct__Foundational_and_structural_provisions']...

  ✓ nodes_hybridity.csv
    Righe: 21  |  Dim: 9.2 KB
    Colonne: ['id', 'hybridity_score', 'hybridity_std', 'hybridity_max', 'most_hybrid_article', 'dominant_lamf']...

  ✓ segments_lamfalussy.csv
    Righe: 906  |  Dim: 1506.3 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'tipo', 'identificatore', 'llm_status']...

  ✓ nodes_lamfalussy.csv
    Righe: 892  |  Dim: 1508.1 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'tipo', 'identificatore', 'llm_status']...

✓

## Fase E — Export per il frontend HTML

Genera `HEATMAPS` (dizionario `id → articoli`) e patcha l'HTML self-contained
con tutti i dati della pipeline: layer labels, distribuzioni percentuali, campo `heatmap` nei nodi.

**Cambia `HTML_FILE`** per puntare al file corretto prima di eseguire.

In [42]:
# ══════════════════════════════════════════════════════════════════════════════
# FASE E — Export HEATMAPS + distribuzione Lamfalussy per il frontend HTML
# ══════════════════════════════════════════════════════════════════════════════

HTML_FILE = os.path.join('..', 'appalti_network.html')   

import re, math, json as _json

# ── 1. Layer labels da layer_mapping.csv ──────────────────────────────────────
layer_df = pd.read_csv(LAYER_MAPPING_FILE).sort_values('layer_rank')
layer_names_raw = layer_df['layer_name'].tolist()

def split_label(name):
    s = name.replace('_', ' ')
    words = s.split()
    if len(words) == 1:
        return s, ''
    half = len(s) // 2
    pos, best_split = 0, len(words) // 2
    for i, w in enumerate(words[:-1]):
        pos += len(w) + 1
        if pos >= half:
            best_split = i + 1
            break
    return ' '.join(words[:best_split]), ' '.join(words[best_split:])

l1_arr, l2_arr = [], []
for name in layer_names_raw:
    a, b = split_label(name)
    l1_arr.append(a)
    l2_arr.append(b)

print(f"Layer ({len(layer_names_raw)}):")
for i, name in enumerate(layer_names_raw):
    print(f"  {i:2d}  '{l1_arr[i]}' / '{l2_arr[i]}'")

# ── Aggiungi lamf per articolo a hm_df ───────────────────────────────────────
hm_df = pd.read_csv(NODES_HEATMAP_FILE)   # ← qui, prima del merge

# ── Aggiungi lamf per articolo a hm_df ───────────────────────────────────────
if os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
    lamf_art = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
    lamf_art = lamf_art[lamf_art['llm_status'] == 'ok'].copy()
    lamf_art = lamf_art.rename(columns={'identificatore': 'articolo_id'})
    if 'node_id' not in lamf_art.columns:
        lamf_art = lamf_art.rename(columns={'celex': 'node_id'})
    hm_df = hm_df.drop(columns=[c for c in hm_df.columns if c.startswith('lamf_L')], errors='ignore')
    lamf_cols_to_add = [c for c in lamf_art.columns if c.startswith('lamf_L')]
    hm_df = hm_df.merge(
        lamf_art[['node_id', 'articolo_id'] + lamf_cols_to_add],
        on=['node_id', 'articolo_id'], how='left'
    )
    print(f"lamf_L1 non-null: {hm_df['lamf_L1'].notna().sum()} / {len(hm_df)}")

# ── 2. HEATMAPS dict da nodes_heatmap.csv ────────────────────────────────────
id_col = 'node_id' if 'node_id' in hm_df.columns else ('id' if 'id' in hm_df.columns else 'celex')
pct_cols_hm = [c for c in hm_df.columns if c.startswith('pct__')]
n_layers = len(pct_cols_hm)

HEATMAPS = {}
for act_id, grp in hm_df.groupby(id_col):
    ok_grp = grp[grp['llm_status'] == 'ok'].copy()
    if ok_grp.empty:
        continue
    articles = []
    for _, row in ok_grp.iterrows():
        vals = [float(row[c]) for c in pct_cols_hm]
        H = 0.0
        for v in vals:
            p = v / 100.0
            if p > 1e-9:
                H -= p * math.log2(p)
        H_norm = round(H / math.log2(n_layers), 3) if n_layers > 1 else 0.0
        
        art_entry = {'id': str(row['articolo_id']), 'H': H_norm,
             'vals': [round(v, 1) for v in vals]}
        # aggiungi lamf se disponibile nel merge
        for lk in ['L1','L2','L3','L4']:
            col = f'lamf_{lk}'
            if col in row and pd.notna(row[col]):
                art_entry.setdefault('lamf', {})[lk] = round(float(row[col]), 1)
        articles.append(art_entry)

    if articles:
        HEATMAPS[str(act_id)] = articles

print(f"\nHEATMAPS: {len(HEATMAPS)} atti  |  "
      f"{sum(len(v) for v in HEATMAPS.values())} articoli totali")

# ── 3. Distribuzione Lamfalussy per atto (media articoli) ─────────────────────
LAMF_DIST = {}
if os.path.exists(NODES_LAMFALUSSY_FILE):
    lamf_df  = pd.read_csv(NODES_LAMFALUSSY_FILE)
    lamf_cols = [c for c in lamf_df.columns if c.startswith('lamf_L')]
    lamf_ok  = lamf_df[lamf_df['llm_status'] == 'ok']
    celex_col = 'celex' if 'celex' in lamf_ok.columns else 'id'
    if lamf_cols and not lamf_ok.empty:
        grp = lamf_ok.groupby(celex_col)[lamf_cols].mean().round(1)
        for act_id, row in grp.iterrows():
            LAMF_DIST[str(act_id)] = {k.replace('lamf_', ''): float(v)
                                       for k, v in row.items()}
    print(f"LAMF_DIST: {len(LAMF_DIST)} atti")
else:
    print("WARN: nodes_lamfalussy.csv non trovato — lamf non aggiunto ai nodi")

# ── 3b. Evidence per articolo da checkpoint A2 ────────────────────────────────
EVIDENCE = {}
if os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
    ckpt = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
    ckpt_ok = ckpt[ckpt['llm_status'] == 'ok']
    ck_id   = 'celex' if 'celex' in ckpt_ok.columns else 'id'
    for (celex, idf), grp in ckpt_ok.groupby([ck_id, 'identificatore']):
        ev_list = []
        for _, row in grp.iterrows():
            raw = row.get('evidence', '[]')
            try:
                ev_list.extend(_json.loads(raw) if isinstance(raw, str) else [])
            except Exception:
                pass
        if ev_list:
            EVIDENCE[(str(celex), str(idf))] = ev_list
    print(f"EVIDENCE: {len(EVIDENCE)} articoli con evidence")
else:
    print("WARN: checkpoint A2 non trovato — evidence non disponibile")

# ── 3c. Testi articoli da nodes_texts.csv ────────────────────────────────────
ART_TEXTS = {}
if os.path.exists(INPUT_FILE):
    texts_df = pd.read_csv(INPUT_FILE)
    id_col_t = 'Id' if 'Id' in texts_df.columns else 'celex'
    for _, row in texts_df.iterrows():
        celex = str(row[id_col_t])
        raw   = row.get('segments', '')
        if not raw or str(raw) in ('nan', '[]', ''):
            continue
        try:
            for seg in _json.loads(str(raw)):
                if seg.get('tipo') == 'articolo':
                    ART_TEXTS[(celex, str(seg.get('identificatore', '')))] = seg.get('testo', '')
        except Exception:
            pass
    print(f"ART_TEXTS: {len(ART_TEXTS)} articoli con testo")

# ── 3d. Arricchisce HEATMAPS con testo ed evidence ───────────────────────────
for celex, articles in HEATMAPS.items():
    for art in articles:
        key = (celex, art['id'])
        art['ev']  = EVIDENCE.get(key, [])
        art['txt'] = ART_TEXTS.get(key, '')

ev_count = sum(1 for arts in HEATMAPS.values() for a in arts if a.get('ev'))
print(f"Articoli con ev in HEATMAPS: {ev_count}")

# ── 4. H Lamfalussy per atto da nodes_hybridity.csv ──────────────────────────
HYB_MAP = {}
if os.path.exists(NODES_HYBRIDITY_FILE):
    hyb_df   = pd.read_csv(NODES_HYBRIDITY_FILE)
    hid_col  = 'celex' if 'celex' in hyb_df.columns else 'id'
    for _, row in hyb_df.iterrows():
        HYB_MAP[str(row[hid_col])] = round(float(row.get('hybridity_score', 0)), 3)
    print(f"HYB_MAP:   {len(HYB_MAP)} atti")

# ── 5. Save heatmaps.json (proper JSON serialization — no syntax errors) ─────
import os as _os

JSON_FILE = _os.path.join(_os.path.dirname(HTML_FILE), 'heatmaps.json')

with open(JSON_FILE, 'w', encoding='utf-8') as f:
    _json.dump(HEATMAPS, f, ensure_ascii=False)

print(f"\n✓ Salvato {JSON_FILE}  ({_os.path.getsize(JSON_FILE)//1024} KB)")

# ── 6. Patch HTML ─────────────────────────────────────────────────────────────
with open(HTML_FILE, 'r', encoding='utf-8') as f:
    html = f.read()

# 6a. Replace inline HEATMAPS const with a fetch()-based async loader.
#     The loader wraps ALL visualization logic that depends on HEATMAPS inside
#     the .then() callback so it runs only after the JSON is available.
FETCH_SNIPPET = (
    "fetch('heatmaps.json')\n"
    "  .then(r => r.json())\n"
    "  .then(HEATMAPS => {\n"
    "    // HEATMAPS loaded from heatmaps.json\n"
    "    window.__HEATMAPS__ = HEATMAPS;\n"
    "  })\n"
    "  .catch(e => console.error('Failed to load heatmaps.json:', e));"
)

# Remove any existing inline HEATMAPS const (may be large, use non-greedy pattern)
html, n_hm = re.subn(
    r'const HEATMAPS\s*=\s*\{[\s\S]*?\};',
    FETCH_SNIPPET,
    html,
    count=1
)
if n_hm == 0:
    # Also try the old HMxxx pattern from prior notebook versions
    html, n_hm = re.subn(r'const HM\w+\s*=\s*\[[\s\S]*?\];', FETCH_SNIPPET, html, count=1)
print(f"HEATMAPS inline → fetch(): {n_hm} sostituzioni")

# Patch all HEATMAPS usages in visualization JS to use window.__HEATMAPS__
html = html.replace('HEATMAPS[', 'window.__HEATMAPS__[')
html = html.replace('HEATMAPS)', 'window.__HEATMAPS__)')
html = html.replace('HEATMAPS,', 'window.__HEATMAPS__,')
html = html.replace('HEATMAPS.', 'window.__HEATMAPS__.')
html = html.replace(' HEATMAPS ', ' window.__HEATMAPS__ ')

# 6b. L1 / L2
html, n = re.subn(r'const L1\s*=\s*\[.*?\];',
                   'const L1     = ' + _json.dumps(l1_arr) + ';', html)
print(f"L1  sostituito: {n}")
html, n = re.subn(r'const L2\s*=\s*\[.*?\];',
                   'const L2     = ' + _json.dumps(l2_arr) + ';', html)
print(f"L2  sostituito: {n}")

# 6c. NODES: heatmap, lamf, H
def patch_nodes(html, heatmap_ids, lamf_dist, hyb_map):
    m = re.search(r'(const NODES\s*=\s*)(\[[\s\S]*?\]);', html)
    if not m:
        print("WARN: const NODES non trovato")
        return html
    nodes = _json.loads(m.group(2))
    for nd in nodes:
        nid = nd['id']
        nd['heatmap'] = 'computed' if nid in heatmap_ids else nd.pop('heatmap', None) or None
        if nd.get('heatmap') is None:
            nd.pop('heatmap', None)
        if nid in lamf_dist:
            nd['lamf'] = lamf_dist[nid]
        else:
            nd.pop('lamf', None)
        if nid in hyb_map:
            nd['H'] = hyb_map[nid]
    new_block = m.group(1) + _json.dumps(nodes, ensure_ascii=False) + ';'
    lamf_n = sum(1 for nd in nodes if 'lamf' in nd)
    hm_n   = sum(1 for nd in nodes if nd.get('heatmap') == 'computed')
    print(f"NODES: {hm_n} con heatmap  |  {lamf_n} con lamf")
    return html[:m.start()] + new_block + html[m.end():]

html = patch_nodes(html, set(HEATMAPS.keys()), LAMF_DIST, HYB_MAP)

with open(HTML_FILE, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"\n✓ {HTML_FILE}")

Layer (6):
   0  'Foundational and structural' / 'provisions'
   1  'Scope and exclusion' / 'clauses'
   2  'Exception and derogation' / 'clauses'
   3  'Implementation and' / 'procedural rules'
   4  'Cross-reference and' / 'incorporation clauses'
   5  'Technical amending' / 'provisions'
lamf_L1 non-null: 892 / 906

HEATMAPS: 21 atti  |  906 articoli totali
LAMF_DIST: 21 atti
EVIDENCE: 892 articoli con evidence
ART_TEXTS: 906 articoli con testo
Articoli con ev in HEATMAPS: 892
HYB_MAP:   21 atti

✓ Salvato ..\heatmaps.json  (4935 KB)
HEATMAPS inline → fetch(): 0 sostituzioni
L1  sostituito: 1
L2  sostituito: 1
NODES: 20 con heatmap  |  20 con lamf

✓ ..\appalti_network.html
